# 01. NumPy and Einsum - 练习

**难度：** Easy | **标签：** `NumPy`, `Broadcasting`, `Einsum` | **目标人群：** Chapter 0 入门学习者

本练习配套导学：[Chapter 0 导学](./intro.md)

## 🎯 学习目标
- 掌握 NumPy 广播与矩阵乘法
- 使用 `einsum` 表达多维张量运算
- 理解 Attention 中的 QK^T 形状推导


## Part 1: NumPy 基础

### 练习 1.1: 构造因果掩码
实现一个函数，返回上三角为 `False`、对角线及下三角为 `True` 的布尔矩阵。


In [ ]:
import numpy as np


def build_causal_mask(seq_len):
    """返回 causal mask，shape = [seq_len, seq_len]。"""
    upper = np.triu(np.ones((seq_len, seq_len), dtype=np.int32), k=1)
    return upper == 0


def matmul_einsum(a, b):
    """使用 einsum 实现二维矩阵乘法。"""
    return np.einsum('ij,jk->ik', a, b)


def batch_attention_scores(q, k):
    """计算 batch attention scores，shape = [b, q, k]。"""
    return np.einsum('bqd,bkd->bqk', q, k)


def rms_normalize(x, eps=1e-6):
    """沿最后一维做 RMSNorm 风格归一化。"""
    denom = np.sqrt(np.mean(np.square(x), axis=-1, keepdims=True) + eps)
    return x / denom


In [ ]:
def test_build_causal_mask():
    mask = build_causal_mask(4)
    expected = np.array([
        [True, False, False, False],
        [True, True, False, False],
        [True, True, True, False],
        [True, True, True, True],
    ])
    assert np.array_equal(mask, expected)
    print('✅ build_causal_mask 通过')


def test_matmul_einsum():
    a = np.array([[1, 2], [3, 4]])
    b = np.array([[5, 6], [7, 8]])
    result = matmul_einsum(a, b)
    expected = np.array([[19, 22], [43, 50]])
    assert np.array_equal(result, expected)
    print('✅ matmul_einsum 通过')


def test_batch_attention_scores():
    q = np.array([[[1, 0], [0, 1]]])
    k = np.array([[[1, 2], [3, 4]]])
    result = batch_attention_scores(q, k)
    expected = np.array([[[1, 3], [2, 4]]])
    assert np.array_equal(result, expected)
    print('✅ batch_attention_scores 通过')


def test_rms_normalize():
    x = np.array([[3.0, 4.0]])
    y = rms_normalize(x, eps=0.0)
    expected = np.array([[0.84852814, 1.13137085]])
    assert np.allclose(y, expected)
    print('✅ rms_normalize 通过')


test_build_causal_mask()
test_matmul_einsum()
test_batch_attention_scores()
test_rms_normalize()


## Part 2: 实战应用

### 练习 2.1: Attention 分数计算
用 `einsum` 计算一个小 batch 的 Attention 分数，观察结果形状。


In [ ]:
q = np.array([[[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]]])
k = np.array([[[1.0, 2.0, 0.0], [3.0, 0.0, 1.0]]])

scores = batch_attention_scores(q, k)
print('Attention scores shape:', scores.shape)
print(scores)

print()
print('Causal mask:')
print(build_causal_mask(5).astype(int))
